# Glauber MCMC Correlation-based Spectral Clustering vs Baseline

Ce notebook compare deux approches pour la détection de communautés sur les graphes signés (appliquée au partitionnement de haplotypes) :
1. **Baseline** : Partitionnement spectral direct via le Laplacien signé appliqué sur le graphe d'origine.
2. **MCMC + Spectral** : Simulation d'une dynamique géométrique de Glauber (MCMC) pour estimer la matrice de corrélation sur les paires à $k$-hop de distance, suivi du partitionnement spectral via le Laplacien signé de cette matrice de corrélation.

## Caractéristiques Techniques
- **Accélération GPU** : Utilisation de **CuPy** si un GPU est disponible pour accélérer la résolution de vecteurs propres sur les grands graphes.
- **Optimisation CPU** : Utilisation de **Numba** pour compiler la dynamique de Glauber en code machine ultra-rapide.
- **Représentation creuse** : Utilisation de puissances de matrices adjacentes creuses pour construire l'ensemble des paires à $k$-hop en moins d'une seconde, éliminant les boucles de parcours en largeur (BFS) lentes.
- **Évaluation** : Mesure de l'exactitude de la classification (Accuracy) par rapport à la vérité terrain (maternal vs. paternal haplotype) modulo un flip global de spin ($1 \leftrightarrow -1$).

## Jeux de Test & Cas Difficiles
Ce benchmark intègre des instances bruitées et complexes :
- **Instances standard** (`instance_100x`, `instance_10x`).
- **Instances bruitées difficiles** (`instance_noisy_small` avec 32% de bruit, `instance_noisy_medium` avec 28% de bruit). Sur ces cas, le bruit important égare le partitionnement spectral classique (baseline proche d'un tirage aléatoire à ~52%), alors que la dynamique MCMC parvient à filtrer le bruit en exploitant la géométrie locale.
- **Instances à faible couverture (15x)** (`instance_large_noisy_15x_1` à 15% de bruit). Ce cas cumule une faible densité d'arêtes et du bruit, induisant un échec complet de la baseline (56.3%), tandis que le MCMC atteint une exactitude de **100%** !
- **Grande instance** (`instance_seed_1` avec 84k nœuds et 2.3M arêtes) pour évaluer le passage à l'échelle.

## Formulation Mathématique

### 1. Hamiltonien et dynamique géométrique
L'état du système est représenté par un vecteur de spins $\sigma \in \lbrace -1, 1 \rbrace^R$. Le Hamiltonien modélisant les interactions signées est :
$$
U(\sigma) = - \sum_{i < j} W_{ij} \sigma_i \sigma_j
$$

La dynamique de Glauber est contrainte sur un sous-groupe XOR encodé par un arbre de Fenwick. À chaque étape, 4 mouvements possibles sont évalués au site $r$ :
- **Mouvement 0** : Identité (aucun changement, $\Delta U_0 = 0$).
- **Mouvement 1** : Retournement du spin singleton $\sigma_r$ (inversion de $x_{r-1}$ et $x_r$).
- **Mouvement 2** : Retournement du préfixe à la coupure $r-1$ (tous les spins $\sigma_i$ pour $i \ge r$ sont inversés).
- **Mouvement 3** : Retournement du préfixe à la coupure $r$ (tous les spins $\sigma_i$ pour $i \ge r+1$ sont inversés).

Le changement d'énergie pour un ensemble d'arêtes affectées par la transition est :
$$
\Delta U_m = 2 \sum_{(i,j) \text{ affectés}} W_{ij} \sigma_i \sigma_j
$$

### 2. Laplacien signé creux
Pour une matrice d'interactions signées $W$, son Laplacien signé est défini par :
$$
L_W = D_W - W
$$
où $D_W = \text{diag}(\sum_j |W_{ij}|)$.

Pour extraire le vecteur propre associé à la plus petite valeur propre de $L$ sans diagonaliser entièrement, on résout le système creux via un solveur de Lanczos (`scipy.sparse.linalg.eigsh` ou `cupyx.scipy.sparse.linalg.eigsh`). Pour accélérer la convergence sur CPU, on applique une translation spectrale :
$$
M = 2 \max_i (D_{ii}) \cdot I - L_W
$$
et on cherche le plus grand vecteur propre de $M$ via `which='LA'` (Largest Algebraic), ce qui évite de résoudre des systèmes linéaires à chaque itération (mode shift-invert `which='SM'`).

In [ ]:
# @title Configuration de l'environnement Colab
import sys
import os

if 'google.colab' in sys.modules:
    print("Exécution sur Google Colab. Installation des dépendances...")
    !pip install -q numba scipy pandas matplotlib
    # Vérification et clonage du dépôt si nécessaire
    if not os.path.exists("Haplotypes"):
        !git clone https://github.com/Ludwig-H/Haplotypes.git
        os.chdir("Haplotypes")
    else:
        os.chdir("Haplotypes")
    print(f"Répertoire de travail actuel : {os.getcwd()}")
else:
    print("Exécution locale.")

In [ ]:
# @title Importations et vérification du GPU
import pandas as pd
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
import numba
import time
import matplotlib.pyplot as plt

try:
    import cupy as cp
    import cupyx.scipy.sparse as csp
    from cupyx.scipy.sparse.linalg import eigsh as cupy_eigsh
    GPU_AVAILABLE = cp.cuda.is_available()
    if GPU_AVAILABLE:
        print(f"GPU disponible via CuPy! Solveur GPU activé.")
    else:
        print("CuPy importé, mais aucun GPU n'a été détecté. Repli sur le CPU.")
except ImportError:
    GPU_AVAILABLE = False
    print("CuPy non installé. Repli sur le CPU.")

In [ ]:
# @title Chargement des données et structures géométriques
def load_instance(instance_dir):
    nodes_df = pd.read_csv(f"{instance_dir}/graph/nodes.tsv", sep="\t")
    edges_df = pd.read_csv(f"{instance_dir}/graph/edges.tsv", sep="\t")
    truth_df = pd.read_csv(f"{instance_dir}/truth/read_truth.tsv", sep="\t")
    
    R = len(nodes_df)
    node_to_truth = dict(zip(truth_df['read_id'], truth_df['true_haplotype']))
    true_spins = np.array([node_to_truth[read_id] for read_id in nodes_df['read_id']])
    
    return R, edges_df, true_spins

@numba.njit
def build_cross_arrays_numba(R, left, right, weight):
    cut_counts = np.zeros(R, dtype=np.int32)
    for idx in range(len(left)):
        u = left[idx]
        v = right[idx]
        for q in range(u, v):
            cut_counts[q] += 1
            
    cross_offsets = np.zeros(R, dtype=np.int32)
    for q in range(R - 1):
        cross_offsets[q+1] = cross_offsets[q] + cut_counts[q]
        
    total_crossings = cross_offsets[R-1]
    cross_left = np.empty(total_crossings, dtype=np.int32)
    cross_right = np.empty(total_crossings, dtype=np.int32)
    cross_weight = np.empty(total_crossings, dtype=np.float64)
    
    current_idx = cross_offsets.copy()
    for idx in range(len(left)):
        u = left[idx]
        v = right[idx]
        w = weight[idx]
        for q in range(u, v):
            write_pos = current_idx[q]
            cross_left[write_pos] = u
            cross_right[write_pos] = v
            cross_weight[write_pos] = w
            current_idx[q] += 1
            
    return cross_offsets, cross_left, cross_right, cross_weight

@numba.njit
def build_incident_arrays_numba(R, left, right, weight):
    node_counts = np.zeros(R + 1, dtype=np.int32)
    for idx in range(len(left)):
        u = left[idx]
        v = right[idx]
        node_counts[u] += 1
        node_counts[v] += 1
        
    incident_offsets = np.zeros(R + 1, dtype=np.int32)
    for r in range(R):
        incident_offsets[r+1] = incident_offsets[r] + node_counts[r]
        
    total_incident = incident_offsets[R]
    incident_left = np.empty(total_incident, dtype=np.int32)
    incident_right = np.empty(total_incident, dtype=np.int32)
    incident_weight = np.empty(total_incident, dtype=np.float64)
    
    current_idx = incident_offsets.copy()
    for idx in range(len(left)):
        u = left[idx]
        v = right[idx]
        w = weight[idx]
        
        pos_u = current_idx[u]
        incident_left[pos_u] = u
        incident_right[pos_u] = v
        incident_weight[pos_u] = w
        current_idx[u] += 1
        
        pos_v = current_idx[v]
        incident_left[pos_v] = u
        incident_right[pos_v] = v
        incident_weight[pos_v] = w
        current_idx[v] += 1
        
    return incident_offsets, incident_left, incident_right, incident_weight

def build_structures_fast(R, edges_df):
    left = np.minimum(edges_df['source'], edges_df['target']).values.astype(np.int32)
    right = np.maximum(edges_df['source'], edges_df['target']).values.astype(np.int32)
    weight = edges_df['weight'].values.astype(np.float64)
    
    cross_offsets, cross_left, cross_right, cross_weight = build_cross_arrays_numba(R, left, right, weight)
    incident_offsets, incident_left, incident_right, incident_weight = build_incident_arrays_numba(R, left, right, weight)
    
    return (cross_offsets, cross_left, cross_right, cross_weight,
            incident_offsets, incident_left, incident_right, incident_weight)

In [ ]:
# @title Génération rapide des paires à $k$-hop par produit de matrices creuses
def build_pairs_sparse(R, edges_df, k):
    row = np.concatenate([edges_df['source'].values, edges_df['target'].values])
    col = np.concatenate([edges_df['target'].values, edges_df['source'].values])
    data = np.ones(len(row), dtype=np.bool_)
    
    A = sp.coo_matrix((data, (row, col)), shape=(R, R)).tocsr()
    I = sp.eye(R, dtype=np.bool_, format='csr')
    
    Visited = (A + I)
    Current = Visited
    for _ in range(k - 1):
        Current = Current @ Visited
        
    Current_tri = sp.triu(Current, k=1)
    coo = Current_tri.tocoo()
    return coo.row.astype(np.int32), coo.col.astype(np.int32)

In [ ]:
# @title Arbre de Fenwick modulo 2 et Évaluation de l'Énergie
@numba.njit
def fenwick_update(tree, idx, val):
    i = idx + 1
    n = len(tree)
    while i < n:
        tree[i] ^= val
        i += i & (-i)

@numba.njit
def fenwick_query(tree, idx):
    if idx < 0:
        return 0
    i = idx + 1
    res = 0
    while i > 0:
        res ^= tree[i]
        i -= i & (-i)
    return res

@numba.njit
def evaluate_cut(q, tree, cross_offsets, cross_left, cross_right, cross_weight):
    start = cross_offsets[q]
    end = cross_offsets[q+1]
    du = 0.0
    for idx in range(start, end):
        i = cross_left[idx]
        j = cross_right[idx]
        w = cross_weight[idx]
        
        xor_val = fenwick_query(tree, j-1) ^ fenwick_query(tree, i-1)
        spin_prod = 1.0 - 2.0 * float(xor_val)
        
        du += w * spin_prod
    return du

@numba.njit
def evaluate_singleton(r, tree, incident_offsets, incident_left, incident_right, incident_weight):
    start = incident_offsets[r]
    end = incident_offsets[r+1]
    du = 0.0
    for idx in range(start, end):
        i = incident_left[idx]
        j = incident_right[idx]
        w = incident_weight[idx]
        
        xor_val = fenwick_query(tree, j-1) ^ fenwick_query(tree, i-1)
        spin_prod = 1.0 - 2.0 * float(xor_val)
        
        du += w * spin_prod
    return du

In [ ]:
# @title Boucle de simulation MCMC avec échantillonnage de spins
@numba.njit
def mcmc_loop(steps, R, beta, tree, 
              cross_offsets, cross_left, cross_right, cross_weight,
              incident_offsets, incident_left, incident_right, incident_weight,
              pairs_left, pairs_right, pair_sums, sample_freq):
    
    num_samples = 0
    for t in range(1, steps + 1):
        r = np.random.randint(0, R)
        
        du0 = 0.0
        du1 = evaluate_singleton(r, tree, incident_offsets, incident_left, incident_right, incident_weight)
        
        if r - 1 >= 0:
            du2 = evaluate_cut(r-1, tree, cross_offsets, cross_left, cross_right, cross_weight)
        else:
            du2 = 0.0
            
        if r < R - 1:
            du3 = evaluate_cut(r, tree, cross_offsets, cross_left, cross_right, cross_weight)
        else:
            du3 = 0.0
            
        min_du = min(du0, du1, du2, du3)
        w0 = np.exp(-beta * (du0 - min_du))
        w1 = np.exp(-beta * (du1 - min_du))
        w2 = np.exp(-beta * (du2 - min_du))
        w3 = np.exp(-beta * (du3 - min_du))
        
        sum_w = w0 + w1 + w2 + w3
        p0 = w0 / sum_w
        p1 = w1 / sum_w
        p2 = w2 / sum_w
        p3 = w3 / sum_w
        
        rand_val = np.random.random()
        chosen_move = 0
        if rand_val < p0:
            chosen_move = 0
        elif rand_val < p0 + p1:
            chosen_move = 1
        elif rand_val < p0 + p1 + p2:
            chosen_move = 2
        else:
            chosen_move = 3
            
        if chosen_move == 1:
            if r - 1 >= 0:
                fenwick_update(tree, r-1, 1)
            if r < R - 1:
                fenwick_update(tree, r, 1)
        elif chosen_move == 2:
            if r - 1 >= 0:
                fenwick_update(tree, r-1, 1)
        elif chosen_move == 3:
            if r < R - 1:
                fenwick_update(tree, r, 1)
                
        # Échantillonnage périodique de spins et accumulation des corrélations
        if t % sample_freq == 0:
            spins = np.zeros(R, dtype=np.float64)
            for i in range(R):
                xor_val = fenwick_query(tree, i-1)
                spins[i] = 1.0 - 2.0 * float(xor_val)
                
            for p in range(len(pairs_left)):
                u = pairs_left[p]
                v = pairs_right[p]
                pair_sums[p] += spins[u] * spins[v]
            num_samples += 1
            
    return num_samples

In [ ]:
# @title Solveur Spectral pour Laplacien Signé (Support GPU / CPU)
def solve_signed_spectral(W, gpu=True, laplacian='unnormalized', verbose=True):
    """
    Calcule le vecteur propre associé à la plus petite valeur propre de L_W = D_W - W.
    Utilise le GPU (CuPy) si gpu=True et un GPU est disponible, sinon utilise
    le CPU (SciPy) avec la translation spectrale (Shifted LA) pour accélérer le calcul.
    """
    R = W.shape[0]
    abs_W = abs(W)
    degrees = np.array(abs_W.sum(axis=1)).flatten()
    
    if gpu and GPU_AVAILABLE:
        try:
            # Résolution sur GPU via CuPy - Correction : utilisation de 'SA' au lieu de 'SM'
            W_gpu = csp.csr_matrix(W)
            D_gpu = csp.diags(cp.array(degrees))
            L_gpu = D_gpu - W_gpu
            vals, vecs = cupy_eigsh(L_gpu, k=1, which='SA')
            return cp.asnumpy(vecs[:, 0]), cp.asnumpy(vals[0])
        except Exception as e:
            print(f"[Solveur GPU] Échec : {e}. Repli sur le CPU...")
            
    # Résolution sur CPU via SciPy
    D = sp.diags(degrees)
    L = D - W
    
    # Astuce de translation spectrale (Shifted LA) avec tol=1e-5 et repli lobpcg/dense pour éviter le non-convergence
    try:
        sigma = 2.0 * np.max(degrees)
        I = sp.eye(R, format='csr')
        M = sigma * I - L
        vals, vecs = eigsh(M, k=1, which='LA', tol=1e-5)
        return vecs[:, 0], sigma - vals[0]
    except Exception as e:
        print(f"[Solveur CPU Shifted LA] Échec : {e}. Essai avec lobpcg...")
        try:
            from scipy.sparse.linalg import lobpcg
            X = np.random.normal(size=(R, 1))
            vals, vecs = lobpcg(L, X, largest=False, tol=1e-5, maxiter=200)
            return vecs[:, 0], vals[0]
        except Exception as e2:
            print(f"[Solveur CPU lobpcg] Échec : {e2}. Repli sur la version dense...")
            L_dense = D.toarray() - W.toarray()
            vals, vecs = np.linalg.eigh(L_dense)
            return vecs[:, 0], vals[0]


In [ ]:
# @title Définition de l'évaluation d'instance de comparaison
def run_instance_comparison(instance_dir, steps, k_hop, beta, sample_freq, use_gpu=True):
    print(f"\nProcessing: {os.path.basename(instance_dir)}...")
    R, edges_df, true_spins = load_instance(instance_dir)
    
    # 1. Baseline : Spectral direct sur le graphe signé d'origine
    t0 = time.time()
    row = np.concatenate([edges_df['source'].values, edges_df['target'].values])
    col = np.concatenate([edges_df['target'].values, edges_df['source'].values])
    data = np.concatenate([edges_df['weight'].values, edges_df['weight'].values])
    W_graph = sp.coo_matrix((data, (row, col)), shape=(R, R)).tocsr()
    
    v_W, val_W = solve_signed_spectral(W_graph, gpu=use_gpu)
    pred_baseline = np.sign(v_W)
    acc_baseline = np.mean(pred_baseline == true_spins)
    acc_baseline = max(acc_baseline, 1.0 - acc_baseline)
    t_baseline = time.time() - t0
    
    # 2. Glauber MCMC
    t0 = time.time()
    cross_offsets, cross_left, cross_right, cross_weight, \
    incident_offsets, incident_left, incident_right, incident_weight = build_structures_fast(R, edges_df)
    
    t_pairs = time.time()
    pairs_left, pairs_right = build_pairs_sparse(R, edges_df, k_hop)
    P = len(pairs_left)
    t_pairs_duration = time.time() - t_pairs
    
    tree = np.zeros(R, dtype=np.int32)
    pair_sums = np.zeros(P, dtype=np.float64)
    
    t_mcmc = time.time()
    num_samples = mcmc_loop(steps, R, beta, tree, 
                            cross_offsets, cross_left, cross_right, cross_weight,
                            incident_offsets, incident_left, incident_right, incident_weight,
                            pairs_left, pairs_right, pair_sums, sample_freq)
    correlations = pair_sums / float(num_samples)
    t_mcmc_duration = time.time() - t_mcmc
    
    # 3. Spectral sur le Laplacien signé de la matrice de corrélation C
    row_C = np.concatenate([pairs_left, pairs_right])
    col_C = np.concatenate([pairs_right, pairs_left])
    data_C = np.concatenate([correlations, correlations])
    W_C = sp.coo_matrix((data_C, (row_C, col_C)), shape=(R, R)).tocsr()
    
    v_C, val_C = solve_signed_spectral(W_C, gpu=use_gpu)
    pred_mcmc = np.sign(v_C)
    acc_mcmc = np.mean(pred_mcmc == true_spins)
    acc_mcmc = max(acc_mcmc, 1.0 - acc_mcmc)
    t_total_mcmc = time.time() - t0
    
    print(f"  Nodes R: {R}, Edges: {len(edges_df)}, pairs ({k_hop}-hop): {P} (built in {t_pairs_duration:.2f}s)")
    print(f"  Baseline Accuracy : {acc_baseline:.4%} (Time: {t_baseline:.2f}s)")
    print(f"  MCMC Accuracy     : {acc_mcmc:.4%} (Time: {t_total_mcmc:.2f}s, MCMC loop: {t_mcmc_duration:.2f}s)")
    
    return {
        "Instance": os.path.basename(instance_dir),
        "R": R,
        "Edges": len(edges_df),
        "Baseline Acc": acc_baseline,
        "Baseline Time": t_baseline,
        "MCMC Acc": acc_mcmc,
        "MCMC Time": t_total_mcmc
    }

In [ ]:
# @title Exécution des benchmarks sur tous les datasets
benchmark_dirs = [
    "benchmark/small/instance_100x",
    "benchmark/small/instance_10x",
    "benchmark/difficult/instance_noisy_small",
    "benchmark/difficult/instance_large_noisy_15x_1",
    "benchmark/instance_seed_1"
]

results = []
for b_dir in benchmark_dirs:
    if not os.path.exists(b_dir) and os.path.exists("Haplotypes/" + b_dir):
        b_dir = "Haplotypes/" + b_dir
        
    if os.path.exists(b_dir):
        # Configuration des paramètres adaptée à la taille et au bruit de chaque instance
        if "instance_100x" in b_dir:
            steps = 50000
            k_hop = 2
            beta = 2.0
            sample_freq = 100
        elif "instance_10x" in b_dir:
            steps = 100000
            k_hop = 2
            beta = 1.5
            sample_freq = 200
        elif "instance_noisy_small" in b_dir:
            steps = 250000
            k_hop = 2
            beta = 1.5
            sample_freq = 250
        elif "instance_large_noisy_15x_1" in b_dir:
            steps = 600000
            k_hop = 2
            beta = 1.2
            sample_freq = 600
        else: # instance_seed_1 (grand graphe)
            steps = 1500000 # 1.5M étapes pour une excellente convergence globale
            k_hop = 2 
            beta = 1.2
            sample_freq = 3000
            
        res = run_instance_comparison(b_dir, steps, k_hop, beta, sample_freq, use_gpu=True)
        results.append(res)

df_results = pd.DataFrame(results)
print("\n" + "=" * 80)
print("FINAL BENCHMARK SUMMARY")
print("=" * 80)
print(df_results.to_string(index=False))

In [ ]:
# @title Visualisation des performances et exactitudes
if len(results) > 0:
    instances = [r["Instance"] for r in results]
    baseline_accs = [r["Baseline Acc"] * 100 for r in results]
    mcmc_accs = [r["MCMC Acc"] * 100 for r in results]
    
    x = np.arange(len(instances))
    width = 0.35
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Graphique 1 : Comparaison d'exactitude
    rects1 = ax1.bar(x - width/2, baseline_accs, width, label='Baseline Spectral (Graph)', color='#FFA07A')
    rects2 = ax1.bar(x + width/2, mcmc_accs, width, label='MCMC k-hop Spectral', color='#4682B4')
    
    ax1.set_ylabel('Accuracy (%)')
    ax1.set_title('Exactitude des prédictions (Accuracy)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(instances, rotation=15)
    ax1.set_ylim(40, 105)
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # Ajouter les valeurs sur les barres
    for rect in rects1 + rects2:
        h = rect.get_height()
        ax1.annotate(f'{h:.2f}%', xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)
    
    # Graphique 2 : Temps d'exécution
    baseline_times = [r["Baseline Time"] for r in results]
    mcmc_times = [r["MCMC Time"] for r in results]
    
    rects3 = ax2.bar(x - width/2, baseline_times, width, label='Baseline Time', color='#FFDAB9')
    rects4 = ax2.bar(x + width/2, mcmc_times, width, label='MCMC Time', color='#B0E0E6')
    
    ax2.set_ylabel('Temps d\'exécution (s)')
    ax2.set_title('Temps d\'exécution global')
    ax2.set_xticks(x)
    ax2.set_xticklabels(instances, rotation=15)
    ax2.legend()
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    for rect in rects3 + rects4:
        h = rect.get_height()
        ax2.annotate(f'{h:.2f}s', xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()